<a href="https://colab.research.google.com/github/Segn11/datasciencebootcamp_project/blob/srypto_pred3/srypoto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [38]:
df= pd.read_csv("/content/Train (8).csv")
test=pd.read_csv("/content/Test (7).csv")

In [39]:
df=df.fillna(0)
test=test.fillna(0)

In [40]:
df.duplicated().any(), test.duplicated().any()

(np.False_, np.False_)

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12632 entries, 0 to 12631
Data columns (total 49 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       12632 non-null  object 
 1   asset_id                 12632 non-null  int64  
 2   open                     12632 non-null  float64
 3   high                     12632 non-null  float64
 4   low                      12632 non-null  float64
 5   volume                   12632 non-null  float64
 6   market_cap               12632 non-null  float64
 7   url_shares               12632 non-null  float64
 8   unique_url_shares        12632 non-null  float64
 9   reddit_posts             12632 non-null  float64
 10  reddit_posts_score       12632 non-null  float64
 11  reddit_comments          12632 non-null  float64
 12  reddit_comments_score    12632 non-null  float64
 13  tweets                   12632 non-null  float64
 14  tweet_spam            

In [42]:
df['H-L'] = df['high'] - df['low']
df['HL_percent'] = (df['high'] - df['low']) / df['low']
df['vol_ratio'] = df['volume'] / df['market_cap']


In [43]:
test['H-L'] = test['high'] - test['low']
test['HL_percent'] = (test['high'] - test['low']) / test['low']
test['vol_ratio'] = test['volume'] / test['market_cap']

In [44]:
features = [
    'open', 'high', 'low', 'volume', 'market_cap', 'market_cap_global',
    'volatility', 'percent_change_24h',
    'H-L', 'HL_percent', 'vol_ratio',
    'tweets', 'social_score', 'social_volume'
]
target = ['close']

In [45]:
import numpy as np

# Replace inf / -inf
df.replace([np.inf, -np.inf], np.nan, inplace=True)
test.replace([np.inf, -np.inf], np.nan, inplace=True)

# Replace 0 in denominators to avoid infinity
df['low'].replace(0, np.nan, inplace=True)
df['market_cap'].replace(0, np.nan, inplace=True)

test['low'].replace(0, np.nan, inplace=True)
test['market_cap'].replace(0, np.nan, inplace=True)

# Fill missing values
df.fillna(0, inplace=True)
test.fillna(0, inplace=True)

#
# Make sure all features are numeric
df[features] = df[features].apply(pd.to_numeric, errors='coerce')
df[features].fillna(0, inplace=True)

test[features] = test[features].apply(pd.to_numeric, errors='coerce')
test[features].fillna(0, inplace=True)

/tmp/ipython-input-1784840506.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['low'].replace(0, np.nan, inplace=True)
/tmp/ipython-input-1784840506.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'd

In [34]:
#features = ['open', 'high', 'low', 'market_cap', 'market_cap_global', 'volume']
#target = ['close']

In [35]:
df=df.fillna(0)
test=test.fillna(0)

In [46]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

estimator = RandomForestRegressor()
selector = RFE(estimator, n_features_to_select=5, step=1)
selector = selector.fit(df[features], df[target])
features_selected = selector.support_

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example usi

In [47]:
new_features = list(np.array(features)[features_selected])
new_features

[np.str_('open'),
 np.str_('high'),
 np.str_('low'),
 np.str_('market_cap'),
 np.str_('market_cap_global')]

In [49]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- Step 2: Prepare train/validation sets ---
X = df[new_features]
y = df[target]  # Replace 'target' with your actual target column

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Step 3: Initialize RandomForest ---
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# --- Step 4: Train the model ---
rf.fit(X_train, y_train)

# --- Step 5: Predict and evaluate ---
y_pred = rf.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

# --- Step 6: Predict on test set ---
X_test = test[new_features]
test_predictions = rf.predict(X_test)

# Optional: add predictions to test dataframe
test['prediction'] = test_predictions

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


MAE: 19.58022592608687
RMSE: 56.21694068329679


In [54]:
y_train_pred = rf.predict(X_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
print("Train MAE:", mae_train)
print("Train RMSE:", rmse_train)

Train MAE: 11.905256098592806
Train RMSE: 25.84890667955301


In [50]:
# --- Step 1: Predict on test set ---
X_test = test[new_features]  # Use the same features selected by RFE
y_p_test = rf.predict(X_test)  # or best_rf if you did hyperparameter tuning

# --- Step 2: Create submission DataFrame ---
# Replace 'ID' with the actual ID column in your test set
submission = pd.DataFrame({
    "ID": test['id'],  # or 'test_id' if that's your column
    "close": y_p_test
})

# --- Step 3: Save to CSV ---
submission.to_csv("submission.csv", index=False)
print("✅ Submission saved!")


✅ Submission saved!


In [51]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [65]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

# -------------------------
# Step 1: Load and clean data
# -------------------------
df = pd.read_csv("/content/Train (8).csv")
test = pd.read_csv("/content/Test (7).csv")

df = df.fillna(0)
test = test.fillna(0)

features = [
    'open', 'high', 'low', 'market_cap', 'market_cap_global'
]
target = 'close'

# Replace inf / -inf
df.replace([np.inf, -np.inf], np.nan, inplace=True)
test.replace([np.inf, -np.inf], np.nan, inplace=True)

# Avoid division by zero
df['low'].replace(0, np.nan, inplace=True)
df['market_cap'].replace(0, np.nan, inplace=True)
test['low'].replace(0, np.nan, inplace=True)
test['market_cap'].replace(0, np.nan, inplace=True)

df.fillna(0, inplace=True)
test.fillna(0, inplace=True)

# Ensure all features are numeric
df[features] = df[features].apply(pd.to_numeric, errors='coerce').fillna(0)
test[features] = test[features].apply(pd.to_numeric, errors='coerce').fillna(0)

# -------------------------
# Step 2: Train-validation split
# -------------------------
X = df[features]
y = df[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------
# Step 3: Hyperparameter optimization
# -------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'max_depth': [8, 10, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# Randomized Search CV
rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=20,
    cv=3,
    scoring='neg_mean_absolute_error',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

rf_random.fit(X_train, y_train)

print("Best Hyperparameters:", rf_random.best_params_)

# -------------------------
# Step 4: Train final model with best params
# -------------------------
best_rf = rf_random.best_estimator_
best_rf.fit(X_train, y_train)

# -------------------------
# Step 5: Predict and evaluate
# -------------------------
y_val_pred = best_rf.predict(X_val)
mae_val = mean_absolute_error(y_val, y_val_pred)
rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))

print(f"Validation MAE: {mae_val:.3f}")
print(f"Validation RMSE: {rmse_val:.3f}")

# -------------------------
# Step 6: Predict on test set
# -------------------------
X_test = test[features]
test_predictions = best_rf.predict(X_test)
test['prediction'] = test_predictions

# Optional: Save submission
submission = test[['id', 'prediction']]  # replace 'id' with actual ID column
submission.to_csv("submission_optimized.csv", index=False)
print("✅ Submission saved!")


Fitting 3 folds for each of 20 candidates, totalling 60 fits


/tmp/ipython-input-2371768546.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['low'].replace(0, np.nan, inplace=True)
/tmp/ipython-input-2371768546.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 

Best Hyperparameters: {'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': None}
Validation MAE: 17.910
Validation RMSE: 55.036
✅ Submission saved!


In [67]:
from google.colab import files
files.download("submission_optimized.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [66]:
# Predict on training set
y_train_pred = best_rf.predict(X_train)

# Evaluate
mae_train = mean_absolute_error(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))

print("Train MAE:", mae_train)
print("Train RMSE:", rmse_train)


Train MAE: 8.56017783309205
Train RMSE: 25.43155413667456
